# Dubai Real Estate 2025 - Dataset Creation and Initial Filtering
This notebook constructs the baseline 2025 residential transactions dataset from the original Dubai Data & Statistics Establishment (DDSE) data source.
The objective is to extract, filter, and store a manageable, analysis-ready subset focused on Dubai’s residential property market.

## Main Steps
**- Column Reduction:** Redundant, administrative, and irrelevant fields are dropped to streamline subsequent analysis.

**- Data Loading:** The full raw dataset (~900 MB) is read in chunks to handle memory constraints efficiently.

**- Year Filtering:** Only records corresponding to the year 2025 are retained.

**- Deduplication:** Verified no duplicate transactions attending to unique transaction IDs.

**- Property-Type Filtering:** Non-residential entries are excluded; the dataset is restricted to Units and Villas.

**- Preliminary Assessment:** Value distributions, feature completeness and data types are reviewed to guide cleaning strategy.

**- Export:** The resulting filtered subset is saved as transactions_2025.csv, forming the input for subsequent preprocessing stages.

## Outcome:
The exported dataset provides a reliable, size-reduced and structurally coherent foundation for detailed cleaning, ensuring the next stage begins with only relevant residential records.

# Libraries

In [4]:
import pandas as pd # For Data Manipulation

# Data

In [7]:
# Path to the original CSV
file_path = "Transactions.csv"

# 1. Column Reduction
Several columns contain the same information in english, arabic and encoded (e.g. 'property_type'). The project will be performed in english, so the arabic versions are removed to reduce file size. For future predictive models, the variables will be encoded again, but for the initial analysis the english version is kept to allow a better understanding of each feature, hence the encoded features are also initially dropped.

Since this is a transactions database, around 99% of the entries contain no information about renting. The two features associated with rent are therefore also removed.

'Procedure_name' is removed for escaping the scope of this project. 'Master project' is removed for being very similar to 'area_name' and containing almost 20% of null values. 'Building name' is also removed for lacking relevant information and containing 22% of null values. 'Property subtype' is removed since all values are either the respective 'property type' or null.

'Project name' is however initially kept, to try and extract developers information.

In [10]:
# List of columns to remove
columns_to_remove = [
    'procedure_id', 'trans_group_id', 'trans_group_ar',
    'procedure_name_ar', 'property_type_id', 'property_type_ar',
    'property_sub_type_id', 'property_sub_type_en','property_sub_type_ar',
    'property_usage_ar', 'reg_type_id', 'reg_type_ar', 'area_id', 'area_name_ar',
    'building_name_ar', 'project_number', 'project_name_ar',
    'master_project_ar', 'nearest_landmark_ar', 'nearest_metro_ar',
    'nearest_mall_ar', 'rooms_ar',  'procedure_name_en',
    'building_name_en', 'master_project_en', 'rent_value', 'meter_rent_price'
]

# 2. Data Loading
The original file counts with over 1.2M entries and 42 rows. The aim is to reduce the dataset to a more managable size, by loading it in chunks and instantly removing all innecesary columns.

For the initial analysis, only data from 2025 is kept, which still contains over 250k entries, enough for a thorough analysis. For future project expansions (i.e. Time Analysis), entries from previous years will also be included. 

In [17]:
chunk_size = 100_000

chunks_list = []  # list to store filtered chunks
new_rows = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
    # Drop unwanted columns
    chunk_reduced = chunk.drop(columns=columns_to_remove, errors='ignore')
    
    # Convert 'instance_date' to datetime
    chunk_reduced['instance_date'] = pd.to_datetime(chunk_reduced['instance_date'], dayfirst=True, errors='coerce')
    
    # Keep only transactions from 2025
    chunk_filtered = chunk_reduced[chunk_reduced['instance_date'].dt.year == 2025]
    
    # Update total row count
    new_rows += len(chunk_filtered)
    
    # Append filtered chunk to list
    chunks_list.append(chunk_filtered)

# Concatenate all chunks into a single DataFrame
df_filtered = pd.concat(chunks_list, ignore_index=True)

print(f"Total rows in final DataFrame: {len(df_filtered)}")

Total rows in final DataFrame: 267517


In [19]:
# Creating a copy to ensure no alteration on the original data
df = df_filtered.copy()

In [21]:
df.head(2)

,transaction_id,trans_group_en,instance_date,property_type_en,property_usage_en,reg_type_en,area_name_en,project_name_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,1-11-2025-22014,Sales,2025-05-29,Building,Residential / Commercial,Existing Properties,Al Hamriya,NaN,Dubai International Airport,Burjuman Metro Station,Dubai Mall,NaN,0,5171.22,73500000.0,14213.28,1.0,1.0,0.0
1,3-9-2025-3468,Gifts,2025-06-12,Building,Residential,Existing Properties,Al Satwa,NaN,Burj Khalifa,Trade Centre Metro Station,Dubai Mall,NaN,0,105.29,2000001.0,18995.17,1.0,2.0,0.0


# 3. Deduplication
We mantained 'transaction_id' for the sole porpouse of checking duplicates. Once it is observed that every transactions has a unique ID, this feature is removed.

In [24]:
# Checking for any duplicated entries
df.duplicated().sum()

0

In [26]:
# Dropping the column since it provides no more value
df = df.drop('transaction_id', axis =1)

# 4. Property Type Filtering
For the purpose of this specific project, only residential properties, and more specifically Units and Villas, are kept.

In [29]:
df['property_usage_en'].value_counts(dropna=False)

property_usage_en
Residential                              230881
Commercial                                20876
Other                                      8246
Hospitality                                4212
Industrial                                 1902
Residential / Commercial                    859
Multi-Use                                   447
Agricultural                                 40
Industrial / Commercial                      38
Storage                                      12
Industrial / Commercial / Residential         4
Name: count, dtype: int64

In [31]:
df['property_type_en'].value_counts(dropna=False)

property_type_en
Unit        208189
Land         29648
Villa        28175
Building      1505
Name: count, dtype: int64

In [33]:
# Keeping only residential properties
df = df[df['property_usage_en']=='Residential']

# Keep only Units and Villas
df = df[df['property_type_en'].isin(['Unit', 'Villa'])].copy()

In [35]:
# Dropping the column since it provides no more value
df = df.drop('property_usage_en', axis =1)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 223801 entries, 2 to 267516
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   trans_group_en        223801 non-null  object        
 1   instance_date         223801 non-null  datetime64[ns]
 2   property_type_en      223801 non-null  object        
 3   reg_type_en           223801 non-null  object        
 4   area_name_en          223801 non-null  object        
 5   project_name_en       199544 non-null  object        
 6   nearest_landmark_en   161164 non-null  object        
 7   nearest_metro_en      131330 non-null  object        
 8   nearest_mall_en       128649 non-null  object        
 9   rooms_en              216627 non-null  object        
 10  has_parking           223801 non-null  int64         
 11  procedure_area        223801 non-null  float64       
 12  actual_worth          223801 non-null  float64       
 13  mete

In [39]:
df['property_type_en'].value_counts()

property_type_en
Unit     195908
Villa     27893
Name: count, dtype: int64

# 5. Preliminary Assesment
Quick check of what type of information and how many null values each column has. 

In [42]:
df.columns

Index(['trans_group_en', 'instance_date', 'property_type_en', 'reg_type_en',
       'area_name_en', 'project_name_en', 'nearest_landmark_en',
       'nearest_metro_en', 'nearest_mall_en', 'rooms_en', 'has_parking',
       'procedure_area', 'actual_worth', 'meter_sale_price',
       'no_of_parties_role_1', 'no_of_parties_role_2', 'no_of_parties_role_3'],
      dtype='object')

In [44]:
df['trans_group_en'].value_counts(dropna=False)

trans_group_en
Sales        183423
Mortgages     33591
Gifts          6787
Name: count, dtype: int64

In [46]:
df['property_type_en'].value_counts(dropna=False)

property_type_en
Unit     195908
Villa     27893
Name: count, dtype: int64

In [48]:
df['reg_type_en'].value_counts(dropna=False)

reg_type_en
Off-Plan Properties    132445
Existing Properties     91356
Name: count, dtype: int64

In [50]:
df['area_name_en'].value_counts(dropna=False)

area_name_en
Al Barsha South Fourth    21548
Business Bay              13216
Wadi Al Safa 5            12908
Madinat Al Mataar         11119
Marsa Dubai                9862
                          ...  
Wadi Al Amardi                1
Al Raffa                      1
Al Mararr                     1
Al Baraha                     1
Al Murqabat                   1
Name: count, Length: 126, dtype: int64

In [52]:
df['area_name_en'].isnull().sum()

0

In [54]:
df['nearest_landmark_en'].value_counts(dropna=False)

nearest_landmark_en
NaN                                  62637
Sports City Swimming Academy         48077
Downtown Dubai                       22579
Motor City                           22467
IMG World Adventures                 21608
Burj Al Arab                         14320
Burj Khalifa                          9789
Dubai International Airport           7724
Expo 2020 Site                        7187
Dubai Cycling Course                  3049
Global Village                        1464
Dubai Parks and Resorts               1077
Al Makhtoum International Airport      970
Hamdan Sports Complex                  853
Name: count, dtype: int64

In [56]:
df['nearest_metro_en'].value_counts(dropna=False)

nearest_metro_en
NaN                                     92471
Dubai Internet City                     18650
Business Bay Metro Station              13404
Nakheel Metro Station                   13122
Buj Khalifa Dubai Mall Metro Station    10320
Sharaf Dg Metro Station                  9438
First Abu Dhabi Bank Metro Station       8808
Damac Properties                         8230
Harbour Tower                            5621
Jumeirah Lakes Towers                    5194
Rashidiya Metro Station                  4926
Creek Metro Station                      4613
Ibn Battuta Metro Station                3142
Al Ghubaiba Metro Station                2310
UAE Exchange Metro Station               1913
Jumeirah Beach Residency                 1870
ENERGY Metro Station                     1858
Palm Jumeirah                            1595
Marina Towers                            1586
Marina Mall Metro Station                1543
Al Jadaf Metro Station                   1467
Trade Centre Metr

In [58]:
df['nearest_mall_en'].value_counts(dropna=False)

nearest_mall_en
NaN                     95152
Marina Mall             42960
Dubai Mall              33863
Mall of the Emirates    31575
Ibn-e-Battuta Mall      11240
City Centre Mirdif       9011
Name: count, dtype: int64

In [60]:
df['project_name_en'].nunique()

2384

In [62]:
df['project_name_en'].isnull().sum()

24257

In [64]:
df['rooms_en'].value_counts(dropna=False)

rooms_en
1 B/R          86510
2 B/R          51520
Studio         45410
3 B/R          22781
4 B/R           9133
NaN             7174
5 B/R           1081
PENTHOUSE         94
6 B/R             74
7 B/R             17
Shop               3
10 B/R             2
Single Room        2
Name: count, dtype: int64

In [66]:
df['has_parking'].value_counts(dropna=False)

has_parking
1    192168
0     31633
Name: count, dtype: int64

In [68]:
df['procedure_area'].isnull().sum()

0

In [70]:
df['instance_date'].isnull().sum()

0

In [72]:
df['actual_worth'].isnull().sum()

0

In [72]:
df['meter_sale_price'].isnull().sum()

0

# 6. Export

In [75]:
df.reset_index(drop=True, inplace=True)

In [77]:
df.to_csv('transactions_2025.csv', index=False)